SKIP: interactive analysis notebook (no savefig). Re-export manually as needed.

**Input data not present in repo.** This notebook reads every `*.csv` in its own directory. Place Azure LLM inference trace CSVs (e.g. `AzureLLMInferenceTrace_conv.csv`) at `./` before running.


In [1]:
import glob, os
import pandas as pd

TRACE_DIR = os.path.dirname(os.path.abspath("__file__"))
csv_files = sorted(glob.glob(os.path.join(TRACE_DIR, "*.csv")))

# preload all CSVs with relative time (both arrival and completion)
traces = {}
for f in csv_files:
    df = pd.read_csv(f)
    t0 = df["ArrivalTime"].min()
    df["t_arrive"] = df["ArrivalTime"] - t0
    df["t_complete"] = df["CompletionTime"] - t0
    traces[os.path.basename(f)] = df

def stats(t_start_min=0, t_end_min=3):
    """Show request stats for each trace in [t_start_min, t_end_min)."""
    rows = []
    t0 = t_start_min * 60
    t1 = t_end_min * 60
    duration = t1 - t0
    for name, df in traces.items():
        arrived = df[(df["t_arrive"] >= t0) & (df["t_arrive"] < t1)]
        completed = df[(df["t_complete"] >= t0) & (df["t_complete"] < t1)]
        if arrived.empty and completed.empty:
            continue
        row = {
            "trace": name,
            "window": f"{t_start_min}-{t_end_min} min",
            "arrived": len(arrived),
            "completed": len(completed),
            "arrive_rps": round(len(arrived) / duration, 2) if duration > 0 else 0,
            "complete_rps": round(len(completed) / duration, 2) if duration > 0 else 0,
        }
        target = completed if not completed.empty else arrived
        row.update({
            "avg_latency": round(target["Latency"].mean(), 2),
            "p50_latency": round(target["Latency"].median(), 2),
            "p99_latency": round(target["Latency"].quantile(0.99), 2),
            "avg_ttft": round(target["TTFT"].mean(), 2),
            "avg_tpot": round(target["TPOT"].mean(), 2),
        })
        rows.append(row)
    return pd.DataFrame(rows)

stats(0, 3)

""


In [2]:
stats(0, 15)

""


In [3]:
stats(0, 10)

""


In [4]:
stats(0, 20)

""


In [5]:
stats(0, 25)

""
